<img src="https://s3.amazonaws.com/weclouddata/images/logos/wcd_logo_new_2.png" width="10%">

<h1><center>RAG Exercise 1: Question Answering Assignment</center></h1>

## Questioning Movie Reviews with RAG

In this notebook, we will build a question answering system based on the reviews of the Avengers: Endgame movie.

We will:

1. Parse reviews from a `csv` file
2. Create a Vectorstore from the reviews
3. Create a `RetrievalQA` using the VectorStore

## Install Libraries

Now let's install our prerequisites.

Besides [LangChain](https://docs.langchain.com/docs/) and [OpenAI](https://github.com/openai/openai-python). For the VectorStore, there are a number of different VectorStores, and a number of different strengths and weaknesses to each.

In this notebook, we will be keeping it very simple by leveraging [Facebook AI Similarity Search](https://ai.meta.com/tools/faiss/#:~:text=FAISS%20(Facebook%20AI%20Similarity%20Search,more%20scalable%20similarity%20search%20functions.), or `FAISS`.

In [1]:
!pip install -q langchain langchain-openai langchain-community langchain-classic langgraph faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


Let's set up our OpenAI API key.

In [24]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY")

OPENAI_API_KEY··········


In [28]:
from openai import OpenAI

client = OpenAI()

models = client.models.list()

print("API key is valid")

API key is valid


## Data Exploration

Let's quickly explore our data.

*You need to upload your data if you are using Google Colab*

In [5]:
import pandas as pd

data_df = pd.read_csv('/content/endgame.csv', index_col=0)

data_df

,Review_Date,Author,Rating,Review_Title,Review,Review_Url
0,29 April 2019,MoistMovies,7.0,Not as good as infinity war..\n,But its a pretty good film. A bit of a mess in...,/review/rw4824873/?ref_=tt_urv
1,13 December 2021,ACollegeStudent,8.0,Not as good as infinity war but a great movie\n,Rating: 8.6,/review/rw4824873/?ref_=tt_urv
2,27 April 2019,nickgray-12862,7.0,Emotional but bit messy.\n,"So it all ends. I have loved most of the MCU, ...",/review/rw4824873/?ref_=tt_urv
3,26 November 2021,davyjones-636363,10.0,Crazy in every sense\n,This film is an emotional rollercoaster with s...,/review/rw4824873/?ref_=tt_urv
4,8 May 2019,dhiraj-yahoo,10.0,Perfect ending\n,"After watching Infinity war, I was looking for...",/review/rw4824873/?ref_=tt_urv
...,...,...,...,...,...,...
344,24 April 2019,bluetanker_07,10.0,The best conclusion to a decade of wonders.\n,I have been giving thoughts if I should still ...,/review/rw4824873/?ref_=tt_urv
345,26 April 2019,mohdmuzmilkabir,10.0,Everybody must watch this one in theater\n,"Best marvel movie ever, conclusion of the 21 m...",/review/rw4824873/?ref_=tt_urv
346,2 May 2023,dakotadickenson-83457,10.0,The end of a era\n,Avengers: Endgame changed the film industry as...,/review/rw4824873/?ref_=tt_urv
347,21 January 2022,Thanos_Alfie,10.0,Masterpiece...\n,"""Avengers: Endgame"" is an Action - Drama movie...",/review/rw4824873/?ref_=tt_urv


## Data Parsing

Now let's start parsing our data into a more usable format for LangChain.

We will use  the `CSVLoader` here.

In [6]:
!pip install -q langchain-community

In [7]:
from langchain_community.document_loaders import CSVLoader

/tmp/ipykernel_1546/3306274919.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader


### Mini-task 1
Check the document and implement the code
- [`CSVLoader`](https://python.langchain.com/docs/integrations/document_loaders/csv)

In [9]:
loader = CSVLoader(
    file_path="/content/endgame.csv",
    source_column="Review"
    )

data = loader.load()

In [10]:
print(data)

[Document(metadata={'source': "But its a pretty good film. A bit of a mess in some parts, lacking the cohesive and effortless feel infinity war somehow managed to accomplish. Some silly plot holes and characters that could've been cut (Ahem, captain marvel and thanos). The use of Captain marvel in this film was just ridiculous. Shes there at the start, bails for some reason? And then pops up at the end to serve no purpose but deux ex machina a space ship...", 'row': 0}, page_content=": 0\nReview_Date: 29 April 2019\nAuthor: MoistMovies\nRating: 7\nReview_Title: Not as good as infinity war..\nReview: But its a pretty good film. A bit of a mess in some parts, lacking the cohesive and effortless feel infinity war somehow managed to accomplish. Some silly plot holes and characters that could've been cut (Ahem, captain marvel and thanos). The use of Captain marvel in this film was just ridiculous. Shes there at the start, bails for some reason? And then pops up at the end to serve no purpos

In [11]:
len(data)

349

Now that we have loaded the review information into a loader - we can go ahead and chunk the reviews into more manageable pieces.

We'll use the `RecursiveCharacterTextSplitter` here.

While splitting our text seems like a simple enough task - getting this correct/incorrect can have massive downstream impacts on your application's performance.


### Mini-task 2

Now we want tosplit our documents into 1000 character length chunks, with 100 characters of overlap. Check the documents and finish the codes below.

- [RecursiveCharacterTextSplitter Docs](https://python.langchain.com/docs/modules/data_connection/document_transformers/text_splitters/recursive_text_splitter)
- [Text Splitter Source Code](https://github.com/langchain-ai/langchain/blob/5e9687a196410e9f41ebcd11eb3f2ca13925545b/libs/langchain/langchain/text_splitter.py#L268C18-L268C18)

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    length_function=len
)

In [13]:
documents = text_splitter.split_documents(data)

In [14]:
print(documents)

[Document(metadata={'source': "But its a pretty good film. A bit of a mess in some parts, lacking the cohesive and effortless feel infinity war somehow managed to accomplish. Some silly plot holes and characters that could've been cut (Ahem, captain marvel and thanos). The use of Captain marvel in this film was just ridiculous. Shes there at the start, bails for some reason? And then pops up at the end to serve no purpose but deux ex machina a space ship...", 'row': 0}, page_content=": 0\nReview_Date: 29 April 2019\nAuthor: MoistMovies\nRating: 7\nReview_Title: Not as good as infinity war..\nReview: But its a pretty good film. A bit of a mess in some parts, lacking the cohesive and effortless feel infinity war somehow managed to accomplish. Some silly plot holes and characters that could've been cut (Ahem, captain marvel and thanos). The use of Captain marvel in this film was just ridiculous. Shes there at the start, bails for some reason? And then pops up at the end to serve no purpos

In [15]:
len(documents)

467

Now, we're ready to create our VectorStore!

## Creating an Index

In the context of the LLM application stack, the term 'index' primarily refers to the process of converting structured documents into a format that is conducive to searching, retrieving, and utilizing

Now, We're going to build our VectorStore with the OpenAI embeddings model. Although it's not necessary for this embeddings model to align with our choice of LLM, it is crucial to maintain consistency when embedding both our index and the queries we run against that index

We don't have to worry too much about that in this exercise, but for more complex applications, it's an important consideration.

Here let's use a `CacheBackedEmbeddings` flow to prevent us from re-embedding similar queries over and over again.


### Mini-task 3

Read the following document to complete the codes below.

- [CacheBackedEmbeddings](https://python.langchain.com/docs/modules/data_connection/caching_embeddings)

In [16]:
from langchain_openai import OpenAIEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.storage import LocalFileStore


In [29]:

store = LocalFileStore("./cache/")

core_embeddings_model = OpenAIEmbeddings()

embedder = CacheBackedEmbeddings.from_bytes_store(
    core_embeddings_model,
    store,
    namespace=core_embeddings_model.model
)

vector_store = FAISS.from_documents(
    documents,
    embedder
)

Now we've created the VectorStore.

we can do a quick check by embedding a query and retrieving passages from our reviews that are close to it.

In [30]:
query = "How do you feel about Iron Man in this movie?"
embedding_vector = core_embeddings_model.embed_query(query)
docs = vector_store.similarity_search_by_vector(embedding_vector, k = 4)

for page in docs:
  print(page.page_content)

Don't want to say too much, but shine a light on Robert Downey Jr. who got to perfect his Iron Man persona to the point where Hugh Jackman can't even surpass him. Captain American is giving all us FanBoys everything we ever wanted to see and Thor...underperforms with hilarious results (and hopefully will create 2019's most popular costume for a certain type of man). Loved Mark Rufflo in this film and Thanos baby, still the baddest villain around.
It's fun it's dramatic, it's like Lord of the Rings with one less movie. Three hours well worth the ticket.
Stan Lee would be proud! Nuff said!
Review_Url: /review/rw4824873/?ref_=tt_urv
Review: In these films the desire and there work rate is fantastic, they do so well together and do well as a team. It is fun to watch them that way. It was a bit boring at times though but still very enjoyable to watch. I can't believe Iron-man has died he was so cool and did very well in his own films. Robert Downey Jr as Iron-Man did so well. And watching t

## Building a Retrieval Chain

Now, we will create a Retrieval Chain that enables us to pose semantic queries on our data. This functionality is largely abstracted in LangChain, making it appear quite robust.


**A Basic RetrievalQA Chain**

We plan to use the `return_source_documents=True` feature to guarantee reliable sources for our reviews, allowing end users to confirm the reviews' authenticity independently.

### Mini-task 4

In this notebook, we'll continue to leverage models from OpenAI - this time we'll use the `gpt-3.5-turbo` model to power our RetrievalQAWithSources chain.


Check out the relevant document to finished the codes:
- [`OpenAIChat()`](https://python.langchain.com/docs/modules/model_io/models/chat/)

In [31]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo")

Now, we are set to construct our retrieval chain.

To effectively use our VectorStore within this chain, it needs to be transformed into a retriever. We can simpliy use the `as_retriever()` method to facilitates this transformation.

- Reference: [`as_retriever()`](https://python.langchain.com/docs/modules/data_connection/retrievers/vectorstore)

In [32]:
retriever = vector_store.as_retriever()

### Mini-task 5

Now check the document below to finish our chain.

- [RetrievalQA](https://api.python.langchain.com/en/latest/chains/langchain.chains.retrieval_qa.base.RetrievalQA.html#)

In [51]:
from langchain_classic.chains import RetrievalQA


# handler = StdOutCallbackHandler()

qa_with_sources_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    # callbacks=[handler],
    return_source_documents=True
)

In [52]:
qa_with_sources_chain({"query" : "How do you feel about Iron Man in this movie?"})

/tmp/ipykernel_1546/3073046764.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  qa_with_sources_chain({"query" : "How do you feel about Iron Man in this movie?"})


{'query': 'How do you feel about Iron Man in this movie?',
 'result': "Iron Man, portrayed by Robert Downey Jr., is highly praised in both reviews for his performance in this movie. The first review mentions that Iron Man's death was impactful, and Robert Downey Jr. did exceptionally well in the role. The second review also mentions that Robert Downey Jr.'s portrayal of Iron Man was one of the reviewer's favorite performances ever of the character.",
 'source_documents': [Document(id='1d225c94-19f5-41db-b1ef-53efd40ec078', metadata={'source': "Man! I thought Infinity War was off the hook, but this one...\nA perfect compliment to a two part epic. It was what I thought it was going to be, a lot of talk, a lot of planning, but amazingly it was not boring. Not for one second!\nInfinity War was the moment we were all waiting for and then they tell us they are going to split it into two films. End Game could have gone south with this choice, but it did not. It's always exciting, especally if

In [53]:
qa_with_sources_chain({"query" : "What did the reviewer think of the movie's ending?"})

{'query': "What did the reviewer think of the movie's ending?",
 'result': "The reviewers generally had positive opinions about the movie's ending. PeteThePrimate mentioned that it was a fitting and moving end to the 22 movie story, gollwog expressed that they both laughed and cried during the film and that none of the minutes were wasted, while larshoeijmans emphasized that the ending made all 22 movies worth it.",
 'source_documents': [Document(id='229c1910-73ed-4386-91d5-b25be4427be5', metadata={'source': "A fitting end to a 22 movie story. It is not a full blown action packed Avengers movie, but what it does, and it does it very well, is wrap up everything in close to 3 hours with respect to everything that's gone before it.\nNo spoilers here, but you'll enjoy the action, laugh at the humour and it will (probably) make you sad in parts. Loved the last couple of scenes especially. It's unusual to be able to watch 2 excellent films in 1 week (Joker being the other).", 'row': 237}, pa

Now we have built our movie review question answering application successfully!